# 03 — Feature Engineering
**Purpose:** Transform the cleaned dataset into a model-ready form: encode categoricals, engineer new features, scale numerics, and split into train/test sets.

**Input:** `data/processed/cleaned_listings.csv`
**Output:** `data/processed/train.csv`, `data/processed/test.csv`


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = pd.read_csv("../data/processed/cleaned_listings.csv")
print(df.shape)
df.head()


(12359, 28)


,location,area,price,price_currency,status,new/resale,price_negotiable,description,security_deposit,facing,...,Swimming Pool,Sports Facility,Jogging Track,Landscaped Gardens,locality_score,Car Parking,city,property_type,price_per_sqft,amenity_count
0,Dhakoli,1300.0,2850000.0,INR,Unknown,0.0,0.0,This spacious 2 bhk builder floor is available...,1.0,Unknown,...,0,0,0,0,NaN,NaN,Chandigarh,Builderfloor,2192.31,0
1,Dhakoli,1400.0,3600000.0,INR,Unknown,0.0,0.0,It’s a 3 bhk builder floor situated in Dhakoli...,1.0,Unknown,...,0,0,0,0,NaN,NaN,Chandigarh,Builderfloor,2571.43,0
2,Dhakoli,1350.0,3690000.0,INR,Unknown,0.0,0.0,It has an area of 1350 sqft with a carpet area...,1.0,Northeast,...,0,0,0,0,NaN,NaN,Chandigarh,Builderfloor,2733.33,0
3,New Chandigarh Mohali,1200.0,10000000.0,INR,Unknown,0.0,0.0,It has a salable area of 1200 sqft and is avai...,1.0,Northeast,...,0,0,0,0,NaN,NaN,Chandigarh,Builderfloor,8333.33,0
4,Sunny Enclave,1008.0,3090000.0,INR,Unknown,1.0,0.0,This spacious 2 bhk builder floor is available...,1.0,East,...,0,0,0,0,NaN,NaN,Chandigarh,Builderfloor,3065.48,0


## 1. Define the Prediction Target
Target: `price` (regression). We'll predict listing price from property characteristics — this
is the core signal used later for the "scheme success" price-benchmarking component.


In [2]:
TARGET = "price"

# Drop columns not useful as model features: free text, IDs, raw currency label, already-derived leakage
drop_cols = ["description", "price_currency", "price_per_sqft", "location"]
drop_cols = [c for c in drop_cols if c in df.columns]

model_df = df.drop(columns=drop_cols)

# Impute remaining sparse numeric columns (moderate missingness, e.g. locality_score, Car Parking)
for c in ["locality_score"]:
    if c in model_df.columns and model_df[c].isna().any():
        model_df[c] = model_df[c].fillna(model_df[c].median())

for c in ["Car Parking"]:
    if c in model_df.columns and model_df[c].isna().any():
        model_df[c] = model_df[c].fillna(0)

print("Remaining columns:", model_df.columns.tolist())
print("\nRemaining NaNs:\n", model_df.isna().sum()[model_df.isna().sum() > 0])


Remaining columns: ['area', 'price', 'status', 'new/resale', 'price_negotiable', 'security_deposit', 'facing', 'furnished', 'age of property', 'Lift(s)', 'Full Power Backup', '24 X 7 Security', "Children's play area", 'Club House', 'Gymnasium', 'Swimming Pool', 'Sports Facility', 'Jogging Track', 'Landscaped Gardens', 'locality_score', 'Car Parking', 'city', 'property_type', 'amenity_count']

Remaining NaNs:
 Series([], dtype: int64)


## 2. Feature Engineering
- `age of property`: keep numeric
- `amenity_count`: already engineered in notebook 1
- One-hot encode categorical columns: `city`, `property_type`, `facing`, `status`


In [3]:
categorical_cols = [c for c in ["city", "property_type", "facing", "status"] if c in model_df.columns]
numeric_cols = [c for c in model_df.columns if c not in categorical_cols + [TARGET]]

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)


Categorical columns: ['city', 'property_type', 'facing', 'status']
Numeric columns: ['area', 'new/resale', 'price_negotiable', 'security_deposit', 'furnished', 'age of property', 'Lift(s)', 'Full Power Backup', '24 X 7 Security', "Children's play area", 'Club House', 'Gymnasium', 'Swimming Pool', 'Sports Facility', 'Jogging Track', 'Landscaped Gardens', 'locality_score', 'Car Parking', 'amenity_count']


In [4]:
# One-hot encode categoricals
model_df = pd.get_dummies(model_df, columns=categorical_cols, drop_first=True)
print("Shape after encoding:", model_df.shape)
model_df.head()


Shape after encoding: (12359, 35)


,area,price,new/resale,price_negotiable,security_deposit,furnished,age of property,Lift(s),Full Power Backup,24 X 7 Security,...,facing_North,facing_Northeast,facing_Northwest,facing_South,facing_Southeast,facing_Southwest,facing_Unknown,facing_West,status_1.0,status_Unknown
0,1300.0,2850000.0,0.0,0.0,1.0,0.0,0,0,0,0,...,False,False,False,False,False,False,True,False,False,True
1,1400.0,3600000.0,0.0,0.0,1.0,0.0,0,0,0,0,...,False,False,False,False,False,False,True,False,False,True
2,1350.0,3690000.0,0.0,0.0,1.0,0.0,0,0,0,0,...,False,True,False,False,False,False,False,False,False,True
3,1200.0,10000000.0,0.0,0.0,1.0,1.0,0,0,0,0,...,False,True,False,False,False,False,False,False,False,True
4,1008.0,3090000.0,1.0,0.0,1.0,0.0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,True


## 3. Log-Transform the Target
Price is heavily right-skewed (seen in EDA) — modeling log(price) tends to produce better-behaved regression residuals.

In [5]:
model_df["log_price"] = np.log1p(model_df[TARGET])
model_df.head()


,area,price,new/resale,price_negotiable,security_deposit,furnished,age of property,Lift(s),Full Power Backup,24 X 7 Security,...,facing_Northeast,facing_Northwest,facing_South,facing_Southeast,facing_Southwest,facing_Unknown,facing_West,status_1.0,status_Unknown,log_price
0,1300.0,2850000.0,0.0,0.0,1.0,0.0,0,0,0,0,...,False,False,False,False,False,True,False,False,True,14.862830
1,1400.0,3600000.0,0.0,0.0,1.0,0.0,0,0,0,0,...,False,False,False,False,False,True,False,False,True,15.096445
2,1350.0,3690000.0,0.0,0.0,1.0,0.0,0,0,0,0,...,True,False,False,False,False,False,False,False,True,15.121137
3,1200.0,10000000.0,0.0,0.0,1.0,1.0,0,0,0,0,...,True,False,False,False,False,False,False,False,True,16.118096
4,1008.0,3090000.0,1.0,0.0,1.0,0.0,0,0,0,0,...,False,False,False,False,False,False,False,False,True,14.943682


## 4. Train / Test Split

In [6]:
feature_cols = [c for c in model_df.columns if c not in [TARGET, "log_price"]]

X = model_df[feature_cols]
y_price = model_df[TARGET]
y_log_price = model_df["log_price"]

X_train, X_test, y_train, y_test, y_train_log, y_test_log = train_test_split(
    X, y_price, y_log_price, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (9887, 34)
Test shape: (2472, 34)


## 5. Scale Numeric Features
Fit the scaler on train only, then apply to both — avoids test-set leakage.

In [7]:
num_feature_cols = [c for c in numeric_cols if c in X_train.columns]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_feature_cols] = scaler.fit_transform(X_train[num_feature_cols])
X_test_scaled[num_feature_cols] = scaler.transform(X_test[num_feature_cols])

X_train_scaled.head()


,area,new/resale,price_negotiable,security_deposit,furnished,age of property,Lift(s),Full Power Backup,24 X 7 Security,Children's play area,...,facing_North,facing_Northeast,facing_Northwest,facing_South,facing_Southeast,facing_Southwest,facing_Unknown,facing_West,status_1.0,status_Unknown
4871,-0.092119,0.549187,-0.599354,0.0,-0.229086,-0.150805,1.514059,1.418947,-0.632791,1.074477,...,False,True,False,False,False,False,False,False,False,True
11455,-0.621670,-1.820874,-0.599354,0.0,-0.229086,0.186273,-0.660476,-0.704748,-0.632791,-0.930686,...,False,False,False,False,False,False,False,False,True,False
10305,-0.324170,0.549187,-0.599354,0.0,-0.229086,-0.150805,-0.660476,-0.704748,-0.632791,-0.930686,...,False,False,False,False,False,False,False,False,True,False
5711,-0.948920,0.549187,-0.599354,0.0,-0.229086,-0.150805,1.514059,1.418947,-0.632791,-0.930686,...,False,False,False,False,False,False,False,False,False,True
12198,2.436632,0.549187,-0.599354,0.0,-0.229086,-0.150805,1.514059,1.418947,1.580300,1.074477,...,False,False,False,False,False,False,False,False,False,True


## 6. Save Model-Ready Datasets

In [8]:
train_out = X_train_scaled.copy()
train_out["price"] = y_train.values
train_out["log_price"] = y_train_log.values

test_out = X_test_scaled.copy()
test_out["price"] = y_test.values
test_out["log_price"] = y_test_log.values

train_out.to_csv("../data/processed/train.csv", index=False)
test_out.to_csv("../data/processed/test.csv", index=False)

print("Saved train.csv:", train_out.shape)
print("Saved test.csv:", test_out.shape)


Saved train.csv: (9887, 36)
Saved test.csv: (2472, 36)


In [9]:
import joblib
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(feature_cols, "../models/feature_columns.pkl")
print("Saved scaler and feature column list to models/")


Saved scaler and feature column list to models/


---
**Next notebook:** `04_modeling.ipynb` — train and compare regression models (Linear Regression, Random Forest, Gradient Boosting/XGBoost) to predict price.
